In [9]:
import os
import load_dotenv
from load_dotenv import load_dotenv

# This function will load all the variable from the .env file and 
# make them available in the os.environ dictionary (env variables)
load_dotenv()

if os.environ.get("CLAUDE_API_KEY"):
    print("CLAUDE API KEY has been loaded")
else:
    raise ValueError("CLAUDE API KEY not found")

if os.environ.get("TAVILY_API_KEY"):
    print("TAVILY API KEY has been loaded")
else:
    raise ValueError("TAVILY API KEY not found")

CLAUDE API KEY has been loaded
TAVILY API KEY has been loaded


In [10]:
from langchain_anthropic import ChatAnthropic

llm_anthropic = ChatAnthropic(
    model = os.environ.get("CLAUDE_MODEL"),
    api_key = os.environ.get('CLAUDE_API_KEY'),
    temperature = 0,
    base_url = os.environ.get("CLAUDE_BASE_URL")
)
llm_anthropic

ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-anthropic': '1.5.5'}}, model='vertex_ai.anthropic.claude-opus-4-6', max_tokens=4096, temperature=0.0, anthropic_api_url='https://genai-sharedservice-americas.pwcinternal.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

In [ ]:
# TOOL - 1 [Tavily Search Tool]

from langchain_tavily import TavilySearch
from langchain.tools import tool

@tool
def tavily_tool(query: str) -> str:

    """This tool searches the latest news on Tavily for the given query and returns the results."""

    tavily_tool = TavilySearch(
        api_key = os.environ.get('TAVILY_API_KEY'),
        max_results = 5,
        description = "This is a tavily tool to search the web for current information"
    )

    tavily_tool.invoke("Today's Trending News about Langchain !!")
    
    return tavily_tool.invoke(query)

In [4]:
# TOOL - 2 [Arxiv Query Search Tool]

from langchain_community.tools import ArxivQueryRun, tool
from langchain_community.utilities import ArxivAPIWrapper

@tool
def arxiv_tool(query: str) -> str:

    """ This tool allows you to query the Arxiv database for research papers. """
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    
    return arxiv_query.invoke(query)

In [5]:
# TOOL - 3 [Wikipedia Search Tool]

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query: str):

    """This tool allows you to search Wikipedia for information on a given topic. """

    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

In [6]:
# TOOL 4 - Custom Tool

from langchain.tools import tool

@tool # This decorator automatically converts this function internally into a tool
def personal_info(name: str):

    """ Use this tool to get personal information about Sayani, Alice and Charlie. """
    info = {
        "Sayani": "Sayani is a data scientist currently focusing on building LLM and RAG pipelines.",
        "Alice": "Alice is a data engineer with 4 years of experience",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    return info.get(name, "No information available for this person")

In [7]:
personal_info.invoke("Alice")

'Alice is a data engineer with 4 years of experience'

#### TOOL Binding

In [ ]:
tools = [tavily_tool, arxiv_tool, wiki_tool, personal_info]

llm_with_tools = llm_anthropic.bind_tools(tools)

response = llm_with_tools.invoke("What is the latest news on AI?")
response.tool_calls